# MFD2 Cleaning

This notebook inspects the Moral Foundations Dictionary 2.0 lexicon, parses the category map,
visualizes term coverage by category, and copies the standardized files into `Data/processed/mfd2`.


In [ ]:
from __future__ import annotations

from pathlib import Path
from ast import literal_eval
import csv
import hashlib
import json
import re
import shutil

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("pandas is required to use this cleaning notebook.") from exc

from IPython.display import display



sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate
    return Path.cwd().resolve()


ROOT = find_project_root()
DATA_ROOT = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data"
RAW_DIR = DATA_ROOT / "raw/mfd2"
OUT_DIR = DATA_ROOT / "processed" / "mfd2"
SAVE_OUTPUTS = False

print("Project root:", ROOT)
print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows, fieldnames=None) -> None:
    ensure_dir(path.parent)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: normalize_for_csv(row.get(k)) for k in fieldnames})


def plot_count(series, title: str, top_n: int = 15):
    counts = series.fillna("<missing>").astype(str).value_counts().head(top_n)
    if counts.empty:
        print(f"No values available for {title}")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_text_length(series, title: str):
    lengths = series.fillna("").astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(lengths, bins=30, ax=ax, color="#55A868")
    ax.set_title(title)
    ax.set_xlabel("characters")
    plt.tight_layout()
    plt.show()


In [ ]:
dic_files = sorted(RAW_DIR.glob("*.dic"))
docx_files = sorted(RAW_DIR.glob("*.docx"))
if not dic_files:
    raise FileNotFoundError(f"No .dic file found under {RAW_DIR}")

dic_path = dic_files[0]
print("Dictionary file:", dic_path.name)
print("Summary docx:", docx_files[0].name if docx_files else "<none>")

preview_lines = dic_path.read_text(encoding="utf-8", errors="replace").splitlines()[:20]
print("\n".join(preview_lines))


In [ ]:
lines = dic_path.read_text(encoding="utf-8", errors="replace").splitlines()
categories = {}
entries = []
mode = "categories"
for line in lines:
    line = line.strip()
    if not line:
        continue
    if line == "%":
        mode = "entries" if mode == "categories_done" else "categories_done"
        continue
    if mode in {"categories", "categories_done"} and "\t" in line and line.split("\t", 1)[0].isdigit():
        idx, label = line.split("\t", 1)
        categories[idx] = label
        continue
    if mode == "entries" and "\t" in line:
        term, category_ids = line.split("\t", 1)
        for category_id in category_ids.split():
            entries.append({
                "term": term,
                "category_id": category_id,
                "category": categories.get(category_id, category_id),
            })

parsed_df = pd.DataFrame(entries)
parsed_df["term_length"] = parsed_df["term"].str.len()
print("Parsed entries:", len(parsed_df))
display(parsed_df.head())


In [ ]:
plot_count(parsed_df["category"], "MFD2 terms by category")
plot_text_length(parsed_df["term"], "MFD2 term length distribution")
display(parsed_df.groupby("category").size().reset_index(name="terms").sort_values("terms", ascending=False))


In [ ]:
if SAVE_OUTPUTS:
    ensure_dir(OUT_DIR)
    shutil.copyfile(dic_path, OUT_DIR / "mfd2.dic")
    if docx_files:
        shutil.copyfile(docx_files[0], OUT_DIR / "mfd2_summary.docx")
    print("Copied standardized MFD2 files to", OUT_DIR)
else:
    print("Preview only. Set SAVE_OUTPUTS = True and rerun this cell to copy standardized files.")
